# Azure Public Dataset V2 — Download

Downloads VM trace data from the **Azure 2019 Public Dataset V2**.
Source: https://github.com/Azure/AzurePublicDataset

Run this once before executing `01_convert_to_parquet.ipynb`.

**Download modes:**
| Mode | Contents | Size |
|------|----------|------|
| `core` | vmtable + subscriptions + deployments + supplementary docs | ~421 MB |
| `sample` | core + supplementary + first 3 CPU reading files | ~2.9 GB |
| `full` | core + supplementary + all 195 CPU reading files | ~156 GB |

## 1. Prerequisites

In [1]:
import sys
from pathlib import Path

# Ensure we can import from app.src/
project_root = Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

Project root: D:\Project\Github\FajarLaksono\sembada-cloud


In [2]:
# Check dependencies
deps_ok = True
try:
    import requests
    print(f"requests: {requests.__version__}")
except ImportError:
    print("\u274c requests not installed. Run:  pip install requests")
    deps_ok = False

try:
    from tqdm import tqdm
    print("tqdm: OK")
except ImportError:
    print("\u274c tqdm not installed. Run:  pip install tqdm")
    deps_ok = False

if deps_ok:
    print("\n\u2705 All dependencies available")
else:
    print("\nInstall missing dependencies above and re-run this cell")

requests: 2.34.0
tqdm: OK

✅ All dependencies available


## 2. Download Configuration

Choose the download mode below. `"sample"` is recommended for first-time users.

In [ ]:
from app.src.download_azure_v2 import MODES, MODE_SIZES

# ── Pick your mode ─────────────────────────────────────────────────
MODE = "sample"  # Change to "core" or "full" if needed
# ────────────────────────────────────────────────────────────────────

OUTPUT_DIR = Path("data/staging/azure_dataset_v2")
WORKERS = 4  # Parallel download threads
SEGMENTS = 0  # 0 = auto, N = split large files into N parallel segments

files = MODES[MODE]
print(f"Mode:     {MODE}")
print(f"Size:     {MODE_SIZES[MODE]}")
print(f"Files:    {len(files)}")
print(f"Output:   {OUTPUT_DIR.resolve()}")
print(f"Workers:  {WORKERS}")
print(f"Segments: {SEGMENTS if SEGMENTS else 'auto'}")

Mode:     core
Size:     ~421 MB  (core tables + supplementary docs)
Files:    11
Output:   D:\Project\Github\FajarLaksono\sembada-cloud\data\staging\azure_dataset_v2
Workers:  4
Segments: auto


In [4]:
from app.src.download_azure_v2 import download_all

print("Starting download...")
download_all(files, OUTPUT_DIR, workers=WORKERS, segments=SEGMENTS)
print("\n\u2705 Download complete")

Starting download...

────────────────────────────────────────────────────────────
  Output directory : D:\Project\Github\FajarLaksono\sembada-cloud\data\staging\azure_dataset_v2
  Files to download: 11
  Parallel workers : 4
  Segments per file: auto
────────────────────────────────────────────────────────────



Overall:   0%|                                                                                | 0/11 [00:00<?, ?file/s]

  → Downloading vmtable.csv.gz ( 417.3 MB) in 4 segments
  → Downloading deployments.csv.gz (   1.9 MB)


  → Downloading Azure 2019 Public Dataset V2 - Trace Analysis.ipynb ( 135.4 KB)
  → Downloading subscriptions.csv.gz ( 343.4 KB)


Overall:  18%|███              | 2/11 [00:04<00:37,  4.12s/file, Azure 2019 Public Dataset V2 - Trace Analysis.ipynb ✓]

...

  ✓ Finished memory.txt (  75.0 B)


Overall:  91%|█████████████████████████████████▋   | 10/11 [00:06<00:00,  2.73file/s, deployments/deployments.csv.gz ✓]

  ✓ Finished deployments.csv.gz (   1.9 MB)


Overall: 100%|█████████████████████████████████████████████| 11/11 [11:46<00:00, 64.19s/file, vmtable/vmtable.csv.gz ✓]

  ✓ Finished vmtable.csv.gz ( 417.3 MB)

────────────────────────────────────────────────────────────
  ✓ Completed : 11/11 files
  All files downloaded successfully.
────────────────────────────────────────────────────────────


✅ Download complete


## 3. Verify Downloaded Files

In [5]:
from app.src.download_azure_v2 import CORE_FILES, CPU_FILES, SUPPLEMENTARY_FILES
import os

OUTPUT_DIR = Path("data/staging/azure_dataset_v2")

print("=" * 60)
print("  DOWNLOADED FILES")
print("=" * 60)

categories = [
    ("Core tables", CORE_FILES),
    ("Supplementary", SUPPLEMENTARY_FILES),
]

total_size = 0
missing = []

for label, file_list in categories:
    print(f"\n  {label}:")
    for _, rel_path in file_list:
        f = OUTPUT_DIR / rel_path
        if f.exists():
            size = f.stat().st_size
            total_size += size
            size_str = f"{size / 1024**2:.1f} MB" if size > 1024**2 else f"{size / 1024:.0f} KB"
            print(f"    \u2705 {rel_path:<45s} {size_str:>10s}")
        else:
            missing.append(rel_path)
            print(f"    \u274c {rel_path:<45s} MISSING")

# CPU readings (only count files actually downloaded)
cpu_dir = OUTPUT_DIR / "vm_cpu_readings"
if cpu_dir.exists():
    cpu_files = sorted(cpu_dir.glob("*.csv.gz"))
    cpu_size = sum(f.stat().st_size for f in cpu_files) / 1024**3
    print(f"\n  CPU readings: {len(cpu_files)} files ({cpu_size:.1f} GB)")
    for f in cpu_files[:3]:
        size_mb = f.stat().st_size / 1024**2
        print(f"    \u2705 {f.name:<45s} {size_mb:>8.1f} MB")
    if len(cpu_files) > 3:
        print(f"    ... and {len(cpu_files) - 3} more files")

total_gb = total_size / 1024**3
print(f"\n  Total: {total_gb:.1f} GB")

if missing:
    print(f"\n  \u26a0 {len(missing)} file(s) missing")
    for m in missing:
        print(f"    - {m}")
else:
    print("\n\u2705 All expected files present")

  DOWNLOADED FILES

  Core tables:
    ✅ vmtable/vmtable.csv.gz                          417.3 MB
    ✅ subscriptions/subscriptions.csv.gz                343 KB
    ✅ deployments/deployments.csv.gz                    1.9 MB

  Supplementary:
    ✅ Azure 2019 Public Dataset V2 - Trace Analysis.ipynb     135 KB
    ✅ schema.csv                                          1 KB
    ✅ azure2019_data/category.txt                         0 KB
    ✅ azure2019_data/cores.txt                            0 KB
    ✅ azure2019_data/cpu.txt                              2 KB
    ✅ azure2019_data/deployment.txt                       1 KB
    ✅ azure2019_data/lifetime.txt                        12 KB
    ✅ azure2019_data/memory.txt                           0 KB

  Total: 0.4 GB

✅ All expected files present
